# tabular_adapter_batch_v1

验证第二批 tabular/tree/boosting/ranking adapter：`random_forest_clf`、`hist_gradient_mse`、`extra_trees_rank`、`lightgbm_mse`、`xgboost_mse`、`catboost_rank`。当前环境没有 `xgboost` 和 `catboost`，因此这两个模型必须显式标记为 dependency fallback，而不是继续混入 smoke fallback。

## 实验治理

- Baseline：上一轮只完成 linear/logistic/ridge adapter，其余 tabular 模型仍是 `smoke_adapter`。
- Direct comparison：tree/boosting/ranking adapter 是否进入统一 `fit_predict_model`、artifact 和 matrix manifest 契约。
- Role：diagnose，推进 QuantBench tabular 模型从 smoke fallback 迁移到显式 adapter。
- Target window：2026-01-01 到 2026-06-09，2026-04-30 前训练，之后测试。
- Changed layer：model。
- External reference：QuantBench tabular family，包括 random forest、hist gradient boosting、LightGBM、XGBoost、CatBoost/ranking、ExtraTrees ranking proxy。
- Acceptance：六个模型都有非 smoke adapter 名称；已安装依赖的模型为 `paper_strict_adapter`；缺失依赖的模型为 `dependency_fallback_adapter`；matrix manifest 能汇总混合状态。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_predict_model, model_adapter_name, model_adapter_status

CARD = repo_root / 'research' / 'tabular_adapter_batch_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:6]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08'}

In [2]:
model_names = (
    'random_forest_clf',
    'hist_gradient_mse',
    'extra_trees_rank',
    'lightgbm_mse',
    'xgboost_mse',
    'catboost_rank',
)
expected = {
    'random_forest_clf': ('paper_strict_adapter', 'random_forest_direction_v1'),
    'hist_gradient_mse': ('paper_strict_adapter', 'hist_gradient_mse_v1'),
    'extra_trees_rank': ('paper_strict_adapter', 'extra_trees_rank_proxy_v1'),
    'lightgbm_mse': ('paper_strict_adapter', 'lightgbm_mse_v1'),
    'xgboost_mse': ('dependency_fallback_adapter', 'xgboost_hist_gradient_dependency_fallback_v1'),
    'catboost_rank': ('dependency_fallback_adapter', 'catboost_extra_trees_rank_dependency_fallback_v1'),
}

predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

for name, (status, adapter_name) in expected.items():
    subset = predictions.loc[predictions['model'] == name]
    assert subset['adapter_status'].unique().tolist() == [status]
    assert subset['model_adapter'].unique().tolist() == [adapter_name]
    assert model_adapter_status(name) == status
    assert model_adapter_name(name) == adapter_name

summary_rows = []
group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
for keys, group in predictions.groupby(group_columns, dropna=False):
    summary_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_summary = pd.DataFrame(summary_rows)
adapter_summary.to_csv(CARD / 'adapter_summary.csv', index=False)
adapter_summary


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,catboost_rank,ranking,boosting,dependency_fallback_adapter,catboost_extra_trees_rank_dependency_fallback_v1,26,0.018919,0.253640,-0.216829
1,extra_trees_rank,ranking,tree,paper_strict_adapter,extra_trees_rank_proxy_v1,26,0.018919,0.253640,-0.216829
2,hist_gradient_mse,mse,tree,paper_strict_adapter,hist_gradient_mse_v1,26,0.007237,0.009800,-0.086570
3,lightgbm_mse,mse,boosting,paper_strict_adapter,lightgbm_mse_v1,26,0.009955,0.011443,-0.104887
4,random_forest_clf,classification,tree,paper_strict_adapter,random_forest_direction_v1,26,0.158684,0.147007,-0.024482
5,xgboost_mse,mse,boosting,dependency_fallback_adapter,xgboost_hist_gradient_dependency_fallback_v1,26,0.007237,0.009800,-0.086570


In [3]:
artifact = fit_model_artifact('lightgbm_mse', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'lightgbm_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_metadata = loaded.metadata_payload()
artifact_predictions.to_csv(CARD / 'lightgbm_artifact_predictions.csv', index=False)
(CARD / 'lightgbm_artifact_metadata.json').write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'lightgbm_mse_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['lightgbm_mse_v1']
{
    'artifact_path': str(artifact_path.relative_to(CARD)),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': 'lightgbm_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'lightgbm_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d',
   'vix_level'],
  'objective': 'mse',
  'family': 'boosting',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'lightgbm_mse_v1',
  'target_column': 'target_return',
  'train_rows': 82},
 'prediction_rows': 26}

In [4]:
matrix = run_quantbench_matrix_smoke(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert manifest['metadata']['adapter_status'] == 'mixed_adapters'
assert manifest['metadata']['adapter_statuses'] == ['dependency_fallback_adapter', 'paper_strict_adapter']
matrix.model_comparison.to_csv(CARD / 'matrix_model_comparison.csv', index=False)
{
    'matrix_status': matrix.summary['status'],
    'top_model': matrix.summary['top_model'],
    'adapter_status': manifest['metadata']['adapter_status'],
    'adapter_statuses': manifest['metadata']['adapter_statuses'],
    'model_adapters': manifest['metadata']['model_adapters'],
    'manifest_hash': manifest['manifest_hash'],
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'matrix_status': 'smoke_completed',
 'top_model': 'random_forest_clf',
 'adapter_status': 'mixed_adapters',
 'adapter_statuses': ['dependency_fallback_adapter', 'paper_strict_adapter'],
 'model_adapters': ['catboost_extra_trees_rank_dependency_fallback_v1',
  'extra_trees_rank_proxy_v1',
  'hist_gradient_mse_v1',
  'lightgbm_mse_v1',
  'random_forest_direction_v1',
  'xgboost_hist_gradient_dependency_fallback_v1'],
 'manifest_hash': '308a29a1e46dccd58124046e635a8ac6e44eeb68a4d85a2504d6cf82857e716e'}

In [5]:
analysis_summary = {
    'schema_version': 'tabular_adapter_batch_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_statuses': sorted(predictions['adapter_status'].unique().tolist()),
    'paper_strict_models': sorted(
        predictions.loc[predictions['adapter_status'] == 'paper_strict_adapter', 'model'].unique().tolist()
    ),
    'dependency_fallback_models': sorted(
        predictions.loc[predictions['adapter_status'] == 'dependency_fallback_adapter', 'model'].unique().tolist()
    ),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'artifact_path': 'lightgbm_mse_artifact.joblib',
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'matrix_top_model': matrix.summary['top_model'],
    'matrix_manifest_hash': manifest['manifest_hash'],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'tabular_adapter_batch_review.v1',
 'status': 'completed',
 'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level'],
 'models': ['random_forest_clf',
  'hist_gradient_mse',
  'extra_trees_rank',
  'lightgbm_mse',
  'xgboost_mse',
  'catboost_rank'],
 'adapter_statuses': ['dependency_fallback_adapter', 'paper_strict_adapter'],
 'paper_strict_models': ['extra_trees_rank',
  'hist_gradient_mse',
  'lightgbm_mse',
  'random_forest_clf'],
 'dependency_fallback_models': ['catboost_rank', 'xgboost_mse'],
 'adapter_summary_path': 'adapter_summary.csv',
 'prediction_path': 'adapter_predictions.csv',
 'artifact_path': 'lightgbm_mse_artifact.joblib',
 'matrix_manifest': 'matrix_card/artifact_manifest.json',
 'matrix_top_model': 'random_forest_clf',
 'matrix_manifest_hash': '308a29a1e46dccd58124046e635a8ac6e44eeb68a4d85a2504d6cf82857e716e'}

## 中文结论

- 做了什么：将 `random_forest_clf`、`hist_gradient_mse`、`extra_trees_rank`、`lightgbm_mse` 接入显式 `paper_strict_adapter`；将当前环境缺少依赖的 `xgboost_mse`、`catboost_rank` 从普通 smoke fallback 改为显式 `dependency_fallback_adapter`。
- 核心指标：六个模型的 adapter 状态、adapter 名称、预测均值/波动和预测与目标相关性写入 `adapter_summary.csv`；LightGBM artifact 的 metadata 和预测写入 sidecar。
- regime 差异：本轮小矩阵通过 `matrix_card` 写出 `regime_metrics.csv`、`window_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，证明新 adapter 能进入既有 qlib/regime 分析链路。
- 是否晋升：否。本轮是模型 adapter 框架推进，不是生产策略晋升。
- 风险：`xgboost_mse` 和 `catboost_rank` 仍受环境依赖限制，只是显式 dependency fallback；`extra_trees_rank` 和 `catboost_rank` 当前是 rank proxy，不等同完整论文级 ranking loss。
- 下一步：补齐 XGBoost/CatBoost 依赖或正式替换方案，再开始 sequence/deep 模型的最小可训练 adapter，并保持相同 artifact/manifest/dashboard 契约。